```text
本实验在上一小时结束、该小时记录可用后，预测下一小时的总租借量。

训练集为 2011 年，验证集为 2012 年 1—9 月，测试集为
2012 年 10—12 月；本阶段不评估测试集。

所有方法统一保留目标值和 W1 输入特征均完整的样本，
最终训练样本为 8,171 个，验证样本为 6,516 个。
W0 使用历史需求和日历信息，W1 额外使用上一小时天气。
```

# 一、数据读取

In [1]:
from pathlib import Path
import os

# 先查看这次实际的工作位置
print("启动位置：", Path.cwd())

# 从当前目录或上一级目录寻找项目根目录
project_root = Path.cwd()

if not (project_root / "src" / "1_data_process.py").exists():
    project_root = project_root.parent

if not (project_root / "src" / "1_data_process.py").exists():
    raise FileNotFoundError("没有找到项目根目录，请检查打开的位置")

# 将后续代码的工作位置统一到项目根目录
os.chdir(project_root)

print("项目根目录：", Path.cwd())
print("能找到原始数据吗：", Path("data/hour.csv").exists())
print("能找到清洗程序吗：", Path("src/1_data_process.py").exists())

启动位置： D:\Desktop\大四上\DDA4220\小组作业\DDA4220-git\baseline
项目根目录： D:\Desktop\大四上\DDA4220\小组作业\DDA4220-git
能找到原始数据吗： True
能找到清洗程序吗： True


In [2]:
# 数据清洗
%run src/1_data_process.py 

完成。输出：
  D:\Desktop\大四上\DDA4220\小组作业\DDA4220-git\data\processed\hour_processed.parquet
  D:\Desktop\大四上\DDA4220\小组作业\DDA4220-git\results\tables
  D:\Desktop\大四上\DDA4220\小组作业\DDA4220-git\config\feature_groups.json


In [3]:
import pandas as pd

df = pd.read_parquet("data/processed/hour_processed.parquet")

print("表格的行数和列数：", df.shape)

df[["datetime", "split", "cnt", "cnt_lag1", "cnt_lag168"]].head(8)
#数据读取

表格的行数和列数： (17544, 59)


,datetime,split,cnt,cnt_lag1,cnt_lag168
0,2011-01-01 00:00:00,train,16.0,NaN,NaN
1,2011-01-01 01:00:00,train,40.0,16.0,NaN
2,2011-01-01 02:00:00,train,32.0,40.0,NaN
3,2011-01-01 03:00:00,train,13.0,32.0,NaN
4,2011-01-01 04:00:00,train,1.0,13.0,NaN
5,2011-01-01 05:00:00,train,1.0,1.0,NaN
6,2011-01-01 06:00:00,train,2.0,1.0,NaN
7,2011-01-01 07:00:00,train,3.0,2.0,NaN


In [4]:
import json

with open("config/feature_groups.json", encoding="utf-8") as f:
    fg = json.load(f)

feat_W0 = fg["time"] + fg["demand_lag"]
feat_W1 = feat_W0 + fg["weather_basic"]

print("W0 特征数：", len(feat_W0))
print("W0 特征：", feat_W0)

print("W1 特征数：", len(feat_W1))
print("W1 特征：", feat_W1)
#读取“模型可以看哪些列”的清单

W0 特征数： 12
W0 特征： ['hr_sin', 'hr_cos', 'weekday_sin', 'weekday_cos', 'workingday', 'mnth_sin', 'mnth_cos', 'cnt_lag1', 'cnt_lag2', 'cnt_lag3', 'cnt_lag24', 'cnt_lag168']
W1 特征数： 20
W1 特征： ['hr_sin', 'hr_cos', 'weekday_sin', 'weekday_cos', 'workingday', 'mnth_sin', 'mnth_cos', 'cnt_lag1', 'cnt_lag2', 'cnt_lag3', 'cnt_lag24', 'cnt_lag168', 'temp_lag1', 'atemp_lag1', 'hum_lag1', 'windspeed_lag1', 'weathersit_1_lag1', 'weathersit_2_lag1', 'weathersit_3_lag1', 'weathersit_4_lag1']


In [5]:
# 只取训练和验证部分，测试集暂时留到最后
development = df[df["split"].isin(["train", "val"])].copy()

# 要求正确答案和 W1 的所有输入都不缺失
required_cols = ["cnt"] + feat_W1
usable = development.dropna(subset=required_cols).copy()

# 按队友已经划分好的标签，分别取出两组
train = usable[usable["split"] == "train"].copy()
val = usable[usable["split"] == "val"].copy()

print("筛选前，训练和验证合计：", len(development))
print("筛选后，训练和验证合计：", len(usable))
print("暂时排除的行数：", len(development) - len(usable))
print("训练集样本数：", len(train))
print("验证集样本数：", len(val))

筛选前，训练和验证合计： 15336
筛选后，训练和验证合计： 14687
暂时排除的行数： 649
训练集样本数： 8171
验证集样本数： 6516


# 二、初步Baseline

## 1. Persistence（上一小时预测法）

In [6]:
# 建立一张验证集预测表
persistence_result = val[["datetime", "cnt", "cnt_lag1"]].copy()

# 给列换成更容易理解的名字
persistence_result = persistence_result.rename(columns={
    "cnt": "y_true",
    "cnt_lag1": "y_pred"
})

# 计算每个小时的绝对误差
persistence_result["absolute_error"] = (
    persistence_result["y_pred"] - persistence_result["y_true"]
).abs()

# 查看前 8 个小时
persistence_result.head(8)

,datetime,y_true,y_pred,absolute_error
8760,2012-01-01 00:00:00,48.0,31.0,17.0
8761,2012-01-01 01:00:00,93.0,48.0,45.0
8762,2012-01-01 02:00:00,75.0,93.0,18.0
8763,2012-01-01 03:00:00,52.0,75.0,23.0
8765,2012-01-01 05:00:00,5.0,8.0,3.0
8766,2012-01-01 06:00:00,2.0,5.0,3.0
8767,2012-01-01 07:00:00,7.0,2.0,5.0
8768,2012-01-01 08:00:00,14.0,7.0,7.0


In [7]:
#计算MAE，查看整体平均错多少
persistence_mae = persistence_result["absolute_error"].mean()

print("参与评估的小时数：", len(persistence_result))
print("Persistence 的验证集 MAE：", round(persistence_mae, 2))

参与评估的小时数： 6516
Persistence 的验证集 MAE： 81.74


## 2. Weekly Naive

In [8]:
#加入上一周数据（下一小时需求＝上周同一小时需求）
# 取出真实需求和上周同小时需求
weekly_result = val[["datetime", "cnt", "cnt_lag168"]].copy()

# 把上周同小时需求作为预测值
weekly_result = weekly_result.rename(columns={
    "cnt": "y_true",
    "cnt_lag168": "y_pred"
})

# 计算每小时的绝对误差
weekly_result["absolute_error"] = (
    weekly_result["y_pred"] - weekly_result["y_true"]
).abs()

# 计算平均绝对误差
weekly_mae = weekly_result["absolute_error"].mean()

print("参与评估的小时数：", len(weekly_result))
print("上一小时预测法 MAE：", round(persistence_mae, 2))
print("上周同小时预测法 MAE：", round(weekly_mae, 2))

参与评估的小时数： 6516
上一小时预测法 MAE： 81.74
上周同小时预测法 MAE： 57.63


In [9]:
# 把已有结果整理成表格
baseline_summary = pd.DataFrame({
    "方法": ["Persistence", "Weekly naive"],
    "验证样本数": [len(persistence_result), len(weekly_result)],
    "MAE": [persistence_mae, weekly_mae]
})

# 计算相对于 Persistence，MAE 降低了多少百分比
weekly_improvement = (
    (persistence_mae - weekly_mae) / persistence_mae * 100
)

display(baseline_summary.round(2))
print("Weekly naive 的 MAE 降低比例：",
      round(weekly_improvement, 2), "%")

,方法,验证样本数,MAE
0,Persistence,6516,81.74
1,Weekly naive,6516,57.63


Weekly naive 的 MAE 降低比例： 29.5 %


## 3. Ridge 回归

前两个方法直接复制某一个历史值；Ridge 会从训练数据中学习，如何组合多个线索来预测需求。例如，它可以同时参考上一小时、昨天同小时、上周同小时以及日历信息。

### (1). Ridge W0：不使用天气

In [10]:
# 训练用的线索和正确答案
X_train_W0 = train[feat_W0].copy()
y_train = train["cnt"].copy()

# 验证用的线索和正确答案
X_val_W0 = val[feat_W0].copy()
y_val = val["cnt"].copy()

print("训练输入的行数、列数：", X_train_W0.shape)
print("训练答案数量：", len(y_train))

print("验证输入的行数、列数：", X_val_W0.shape)
print("验证答案数量：", len(y_val))

训练输入的行数、列数： (8171, 12)
训练答案数量： 8171
验证输入的行数、列数： (6516, 12)
验证答案数量： 6516


In [11]:
#让 Ridge W0 从训练数据中学习
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# 建立流程：先标准化输入，再用 Ridge 学习
ridge_W0 = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

# 只用训练集学习
ridge_W0.fit(X_train_W0, y_train)

print("Ridge W0 训练完成")

Ridge W0 训练完成


学习组合：
预测需求 =
上一小时需求的贡献
＋上周同小时需求的贡献
＋日历等其他特征的贡献
＋一个基础值

In [12]:
#用训练好的 Ridge W0 预测验证集，并计算 MAE
# 只把验证集的线索交给模型，生成预测
ridge_W0_pred = ridge_W0.predict(X_val_W0)

# 预测完成后，再用真实答案评分
ridge_W0_mae = (
    (ridge_W0_pred - y_val).abs().mean()
)

print("预测的小时数：", len(ridge_W0_pred))
print("Persistence MAE：", round(persistence_mae, 2))
print("Weekly naive MAE：", round(weekly_mae, 2))
print("Ridge W0 MAE：", round(ridge_W0_mae, 2))

预测的小时数： 6516
Persistence MAE： 81.74
Weekly naive MAE： 57.63
Ridge W0 MAE： 47.76


### (2). 训练 Ridge W1，加入上一小时天气

In [13]:
# 准备加入天气后的输入
X_train_W1 = train[feat_W1].copy()
X_val_W1 = val[feat_W1].copy()

# 与 W0 使用相同的模型设置
ridge_W1 = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

# 在训练集学习
ridge_W1.fit(X_train_W1, y_train)

# 对相同的验证小时生成预测
ridge_W1_pred = ridge_W1.predict(X_val_W1)

# 计算 MAE
ridge_W1_mae = (ridge_W1_pred - y_val).abs().mean()

print("W0 输入特征数：", X_train_W0.shape[1])
print("W1 输入特征数：", X_train_W1.shape[1])
print("Ridge W0 MAE：", round(ridge_W0_mae, 2))
print("Ridge W1 MAE：", round(ridge_W1_mae, 2))

W0 输入特征数： 12
W1 输入特征数： 20
Ridge W0 MAE： 47.76
Ridge W1 MAE： 47.06


## 4. 初步实验对比

### （1）. RMSE：检查大的预测错误

In [14]:
#RMSE 会对较大的错误给予更重的惩罚
import numpy as np
from sklearn.metrics import mean_squared_error

# 用已经得到的预测计算 RMSE，不需要重新训练
persistence_rmse = np.sqrt(mean_squared_error(
    persistence_result["y_true"],
    persistence_result["y_pred"]
))

weekly_rmse = np.sqrt(mean_squared_error(
    weekly_result["y_true"],
    weekly_result["y_pred"]
))

ridge_W0_rmse = np.sqrt(mean_squared_error(
    y_val, ridge_W0_pred
))

ridge_W1_rmse = np.sqrt(mean_squared_error(
    y_val, ridge_W1_pred
))

# 汇总四种方法的结果
baseline_summary = pd.DataFrame({
    "方法": ["Persistence", "Weekly naive", "Ridge W0", "Ridge W1"],
    "验证样本数": [len(val)] * 4,
    "MAE": [
        persistence_mae, weekly_mae,
        ridge_W0_mae, ridge_W1_mae
    ],
    "RMSE": [
        persistence_rmse, weekly_rmse,
        ridge_W0_rmse, ridge_W1_rmse
    ]
})

display(baseline_summary.round(2))

,方法,验证样本数,MAE,RMSE
0,Persistence,6516,81.74,122.53
1,Weekly naive,6516,57.63,99.40
2,Ridge W0,6516,47.76,73.70
3,Ridge W1,6516,47.06,70.98


当前结论：在相同的 6,516 个验证样本上，固定 alpha=1.0，加入上一小时天气后，Ridge 的 MAE 从 47.76 降至 47.06，RMSE 从 73.70 降至 70.98，表明天气信息在当前设置下带来了小幅预测改善。

### (2). $R^2$: 模型相对于“每个小时都猜同一个平均值”，减少了多少平方误差

In [15]:
from sklearn.metrics import r2_score

# 顺序与汇总表中的四种方法一致
baseline_summary["R2"] = [
    r2_score(
        persistence_result["y_true"],
        persistence_result["y_pred"]
    ),
    r2_score(
        weekly_result["y_true"],
        weekly_result["y_pred"]
    ),
    r2_score(y_val, ridge_W0_pred),
    r2_score(y_val, ridge_W1_pred)
]

display(baseline_summary.round(3))

,方法,验证样本数,MAE,RMSE,R2
0,Persistence,6516,81.738,122.531,0.663
1,Weekly naive,6516,57.627,99.399,0.778
2,Ridge W0,6516,47.756,73.696,0.878
3,Ridge W1,6516,47.063,70.975,0.887


# 三、Daily naive：昨天同小时预测法

使用昨天同一小时的真实需求，预测目标小时需求。
本实验使用与其他 baseline 相同的验证样本。

In [16]:
# 用昨天同小时需求生成预测
daily_result = val[["datetime", "cnt", "cnt_lag24"]].copy()

daily_result = daily_result.rename(columns={
    "cnt": "y_true",
    "cnt_lag24": "y_pred"
})

# 计算三个评价指标
daily_mae = (
    daily_result["y_pred"] - daily_result["y_true"]
).abs().mean()

daily_rmse = np.sqrt(mean_squared_error(
    daily_result["y_true"],
    daily_result["y_pred"]
))

daily_r2 = r2_score(
    daily_result["y_true"],
    daily_result["y_pred"]
)

print("验证样本数：", len(daily_result))
print("Daily naive MAE：", round(daily_mae, 3))
print("Daily naive RMSE：", round(daily_rmse, 3))
print("Daily naive R²：", round(daily_r2, 3))

验证样本数： 6516
Daily naive MAE： 77.316
Daily naive RMSE： 129.351
Daily naive R²： 0.624


# 四、Ridge 参数选择

In [17]:
from sklearn.metrics import mean_absolute_error

# 预先规定要尝试的 alpha
alpha_candidates = [0.1, 1.0, 10.0, 100.0]

# 保存每次实验的成绩，以及训练好的模型
ridge_tuning_rows = []
ridge_candidates = {}

# W0 和 W1 使用相同的候选参数、训练样本和验证样本
for group_name, features in [
    ("W0", feat_W0),
    ("W1", feat_W1)
]:
    for alpha in alpha_candidates:

        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=alpha)
        )

        # 只在训练集学习
        model.fit(train[features], y_train)

        # 在验证集评分
        prediction = model.predict(val[features])
        mae = mean_absolute_error(y_val, prediction)

        # 记录这次实验
        ridge_tuning_rows.append({
            "feature_group": group_name,
            "alpha": alpha,
            "val_MAE": mae
        })

        ridge_candidates[(group_name, alpha)] = model

ridge_tuning_table = pd.DataFrame(ridge_tuning_rows)

display(ridge_tuning_table.round(4))

,feature_group,alpha,val_MAE
0,W0,0.1,47.7593
1,W0,1.0,47.7560
2,W0,10.0,47.7250
3,W0,100.0,47.6389
4,W1,0.1,47.0660
5,W1,1.0,47.0627
6,W1,10.0,47.0352
7,W1,100.0,47.0280


说明：本次尝试的参数范围内，调整 alpha 对验证 MAE 的影响有限。

In [18]:
# 保存选中的参数、模型、预测和指标
ridge_selected_alphas = {}
ridge_selected_models = {}
ridge_selected_predictions = {}
ridge_selected_rows = []

for group_name, features in [
    ("W0", feat_W0),
    ("W1", feat_W1)
]:
    # 只查看当前输入组的调参记录
    group_scores = ridge_tuning_table[
        ridge_tuning_table["feature_group"] == group_name
    ]

    # 找到验证 MAE 最低的那一行
    best_index = group_scores["val_MAE"].idxmin()
    best_alpha = float(group_scores.loc[best_index, "alpha"])

    # 取出之前已经训练好的模型，生成验证预测
    selected_model = ridge_candidates[(group_name, best_alpha)]
    selected_prediction = selected_model.predict(val[features])

    ridge_selected_alphas[group_name] = best_alpha
    ridge_selected_models[group_name] = selected_model
    ridge_selected_predictions[group_name] = selected_prediction

    # 记录三个评价指标
    ridge_selected_rows.append({
        "方法": "Ridge selected",
        "输入方案": group_name,
        "alpha": best_alpha,
        "验证样本数": len(val),
        "MAE": mean_absolute_error(y_val, selected_prediction),
        "RMSE": np.sqrt(
            mean_squared_error(y_val, selected_prediction)
        ),
        "R2": r2_score(y_val, selected_prediction)
    })

ridge_selected_summary = pd.DataFrame(ridge_selected_rows)
display(ridge_selected_summary.round(4))

,方法,输入方案,alpha,验证样本数,MAE,RMSE,R2
0,Ridge selected,W0,100.0,6516,47.6389,73.8358,0.8775
1,Ridge selected,W1,100.0,6516,47.0280,71.2219,0.8860


按验证 MAE 选择的参数

# 五、XGBoost 强基线

#若没有安装XGboost，运行安装
%pip install "xgboost>=2.0"

In [19]:
import xgboost as xgb

print("XGBoost 版本：", xgb.__version__)

XGBoost 版本： 3.4.1


## 1. W0：无天气

In [20]:
# 预先固定训练设置，之后 W1 使用完全相同的配置
xgb_params = {
    "objective": "reg:squarederror",
    "n_estimators": 300,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "random_state": 42,
    "n_jobs": 4,
    "tree_method": "hist"
}

# 建立模型
xgb_W0 = xgb.XGBRegressor(**xgb_params)

# 只用训练集学习，不输入天气
xgb_W0.fit(X_train_W0, y_train)

print("XGBoost W0 训练完成")

XGBoost W0 训练完成


In [21]:
# 根据验证集输入生成预测
xgb_W0_pred = xgb_W0.predict(X_val_W0)

# 用真实答案计算三个指标
xgb_W0_mae = mean_absolute_error(y_val, xgb_W0_pred)

xgb_W0_rmse = np.sqrt(
    mean_squared_error(y_val, xgb_W0_pred)
)

xgb_W0_r2 = r2_score(y_val, xgb_W0_pred)

print("预测的小时数：", len(xgb_W0_pred))
print("XGBoost W0 MAE：", round(xgb_W0_mae, 4))
print("XGBoost W0 RMSE：", round(xgb_W0_rmse, 4))
print("XGBoost W0 R²：", round(xgb_W0_r2, 4))

预测的小时数： 6516
XGBoost W0 MAE： 48.2303
XGBoost W0 RMSE： 82.5874
XGBoost W0 R²： 0.8468


## 2. W1：有天气

In [22]:
# 使用与 W0 完全相同的模型设置
xgb_W1 = xgb.XGBRegressor(**xgb_params)

# 只增加上一小时天气输入
xgb_W1.fit(X_train_W1, y_train)

# 对相同的验证小时生成预测
xgb_W1_pred = xgb_W1.predict(X_val_W1)

# 计算三个评价指标
xgb_W1_mae = mean_absolute_error(y_val, xgb_W1_pred)

xgb_W1_rmse = np.sqrt(
    mean_squared_error(y_val, xgb_W1_pred)
)

xgb_W1_r2 = r2_score(y_val, xgb_W1_pred)

# 展示天气对照结果
xgb_summary = pd.DataFrame({
    "输入方案": ["W0：无天气", "W1：有天气"],
    "验证样本数": [len(val), len(val)],
    "MAE": [xgb_W0_mae, xgb_W1_mae],
    "RMSE": [xgb_W0_rmse, xgb_W1_rmse],
    "R2": [xgb_W0_r2, xgb_W1_r2]
})

display(xgb_summary.round(4))

,输入方案,验证样本数,MAE,RMSE,R2
0,W0：无天气,6516,48.2303,82.5874,0.8468
1,W1：有天气,6516,47.3583,81.4720,0.8509


# 六、期中 baseline 完整汇总与导出

In [23]:
# 汇集全部预测，不覆盖前面的实验变量
all_predictions = {
    "Persistence": persistence_result["y_pred"].to_numpy(),
    "Daily_naive": daily_result["y_pred"].to_numpy(),
    "Weekly_naive": weekly_result["y_pred"].to_numpy(),
    "Ridge_W0_alpha1": ridge_W0_pred,
    "Ridge_W1_alpha1": ridge_W1_pred,
    "Ridge_W0_selected": ridge_selected_predictions["W0"],
    "Ridge_W1_selected": ridge_selected_predictions["W1"],
    "XGBoost_W0": xgb_W0_pred,
    "XGBoost_W1": xgb_W1_pred,
}

# 统一重新计算指标，避免手动抄错
final_rows = []

for experiment, prediction in all_predictions.items():
    prediction = np.asarray(prediction)

    # 检查每种方法都有完整、有限的预测值
    assert prediction.shape == (len(val),), experiment
    assert np.isfinite(prediction).all(), experiment

    final_rows.append({
        "experiment": experiment,
        "split": "val",
        "n_samples": len(val),
        "MAE": mean_absolute_error(y_val, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_val, prediction)),
        "R2": r2_score(y_val, prediction)
    })

final_metrics = pd.DataFrame(final_rows)

# 逐小时预测表：每行对应同一个验证小时
final_predictions = val[["datetime", "cnt"]].copy()
final_predictions = final_predictions.rename(
    columns={"cnt": "y_true"}
)

for experiment, prediction in all_predictions.items():
    final_predictions[experiment] = np.asarray(prediction)

# 创建正式交付文件夹
delivery_dir = Path("baseline/results")
delivery_dir.mkdir(parents=True, exist_ok=True)

# 保存完整结果，保留原始精度
final_metrics.to_csv(
    delivery_dir / "metrics_val.csv",
    index=False,
    encoding="utf-8-sig"
)

final_predictions.to_csv(
    delivery_dir / "predictions_val.csv",
    index=False,
    encoding="utf-8-sig"
)

ridge_tuning_table.to_csv(
    delivery_dir / "ridge_tuning_val.csv",
    index=False,
    encoding="utf-8-sig"
)

# 保存训练样本时间戳，方便组员核对样本是否一致
train[["datetime"]].to_csv(
    delivery_dir / "train_timestamps.csv",
    index=False
)

# 保存实验设置与软件版本
import sys
import sklearn

experiment_config = {
    "forecast_timing": "Predict hour t after hour t-1 is observed",
    "sample_policy": "Complete target and all W1 features",
    "n_train": len(train),
    "n_val": len(val),
    "features": {
        "W0": feat_W0,
        "W1": feat_W1
    },
    "ridge": {
        "preprocessing": "StandardScaler fitted on train only",
        "fixed_alpha": 1.0,
        "alpha_candidates": alpha_candidates,
        "selection_metric": "validation MAE",
        "selected_alphas": ridge_selected_alphas
    },
    "xgboost": {
        "preprocessing": "No scaling",
        "parameter_policy": "Fixed configuration, no tuning",
        "params": xgb_params
    },
    "prediction_postprocessing": "None; no clipping or rounding",
    "test_evaluated": False,
    "versions": {
        "python": sys.version,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__
    }
}

with open(
    delivery_dir / "experiment_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(experiment_config, f, ensure_ascii=False, indent=2)

display(final_metrics.round(4))
print("预测表行数：", len(final_predictions))
print("交付文件保存位置：", delivery_dir.resolve())

,experiment,split,n_samples,MAE,RMSE,R2
0,Persistence,val,6516,81.7383,122.5311,0.6627
1,Daily_naive,val,6516,77.3157,129.3510,0.6241
2,Weekly_naive,val,6516,57.6272,99.3994,0.7780
3,Ridge_W0_alpha1,val,6516,47.7560,73.6958,0.8780
4,Ridge_W1_alpha1,val,6516,47.0627,70.9751,0.8868
5,Ridge_W0_selected,val,6516,47.6389,73.8358,0.8775
6,Ridge_W1_selected,val,6516,47.0280,71.2219,0.8860
7,XGBoost_W0,val,6516,48.2303,82.5874,0.8468
8,XGBoost_W1,val,6516,47.3583,81.4720,0.8509


预测表行数： 6516
交付文件保存位置： D:\Desktop\大四上\DDA4220\小组作业\DDA4220-git\baseline\results


# 七、结论

本阶段完成了 Persistence、Daily naive、Weekly naive、Ridge 和 XGBoost 的验证集实验。所有方法使用相同的 **6,516 个验证样本**，并以 MAE 为主要评价指标，RMSE 和 R² 为辅助指标。

- **简单预测方法中，Weekly naive 表现最好**，说明上周同小时需求是一个有效的预测参照。
- **Ridge 和 XGBoost 均优于三个简单方法**。在本次已测试的配置中，Ridge W1 表现最好：按验证 MAE 选参后，MAE 为 **47.028**；固定 `alpha=1` 时，RMSE 为 **70.975**、R² 为 **0.887**。这些指标来自不同的 Ridge 配置，不应合并为同一组实验结果。
- **天气信息带来了小幅改善**。固定参数的 Ridge 和 XGBoost 中，W1 相比 W0 的 MAE 分别降低约 **1.45%** 和 **1.81%**，支持上一小时天气在当前实验设置下提供额外预测信息。
- **更复杂的方法不一定更好**：当前固定配置的 XGBoost 未超过 Ridge，后续 MLP 应同时与简单周期基线、Ridge 和 XGBoost 比较。

以上结论仅适用于当前数据划分、样本规则和已测试配置。验证集参与了 Ridge 参数选择，测试集尚未用于评估；目前不能将天气带来的改善称为统计显著提升。